# Bonus — Testing AI Systems Code
## From an inline `assert` to a project test suite

**Audience:** generalist engineering students with varied backgrounds
**Format:** context → mental model → worked examples → checks → open project
**Prerequisites:** Python functions, exceptions, `asyncio` basics, and any one of the earlier notebooks' projects.

### Learning goals

By the end of this notebook, you should be able to:

- explain what a project test suite must do that a scattered `assert` does not;
- structure a test as **arrange / act / assert**, and run a set of them with a tiny stdlib runner;
- write **table-driven** tests that report *which* case failed;
- test that the right **exception** is raised, and test **floats** and **randomness** without brittle equality;
- test **async** code, including a concurrency ceiling and a retry count;
- keep tests **isolated** from shared state; and
- translate all of the above to `pytest`.

Every instructional example is complete and executable. Only the final project is intentionally unfinished.

> This course has no `pytest` in its environment on purpose: the notebook teaches a ~15-line runner so every cell here runs offline, then shows the `pytest` translation for real projects.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification.

Some cells define functions named `test_...` and run them through `run_tests(...)` (built in section 2). Two of those tests **fail on purpose** — one in section 2 (to show what a failure report looks like) and one in section 7 (to show an order-dependent test). Both are labelled. The notebook still runs all the way through, because `run_tests` catches a failing test instead of stopping, and each cell then checks the *expected* number of passes and failures.

## Before we start: a suite, not scattered asserts

Every notebook in this course ends its cells with `assert`. That is the right tool *in a notebook*: a claim, checked the instant you run the cell, right next to the code.

A **project** needs more. When you hand in the inference gateway (notebook 01) or the resilient client (notebook 05), you — and whoever grades it — need a **test suite**:

| | a few `assert`s in a script | a test suite |
|---|---|---|
| runs with nobody watching | stops at the **first** failure | runs **all** the tests, then reports |
| tells you what broke | `AssertionError`, no context | names the test and the input |
| covers cases on purpose | whatever you happened to type | a planned list of inputs |
| survives you changing the code | asserts get deleted | the tests are the spec |
| keeps tests independent | one test's leftovers break the next | each test starts clean |

The rest of the notebook builds each of these with the standard library, then shows the `pytest` version. The ideas are the same in any framework.

## 1. Anatomy of a test: arrange, act, assert

A test is three steps with a blank line between them:

1. **Arrange** — build the inputs and the expected result.
2. **Act** — call the code under test, once.
3. **Assert** — compare the actual result to the expected one.

The function under test below, `to_probability`, is the boundary-conversion pattern from notebook 01: parse a value, enforce the domain invariant `0 <= p <= 1`, and reject everything else with a clear error.

**Predict** which of `"0.5"`, `1`, `"nan"`, `2.0` and `None` `to_probability` accepts.

In [1]:
import math

def to_probability(value: object) -> float:
    """Convert `value` to a float in [0, 1]. Raise ValueError otherwise (nan is rejected)."""
    try:
        p = float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        raise ValueError(f"not a number: {value!r}") from None
    if not (0.0 <= p <= 1.0):        # nan fails this: every comparison with nan is False
        raise ValueError(f"outside [0, 1]: {p!r}")
    return p


def test_accepts_a_valid_string() -> None:
    expected = 0.5                                  # arrange
    result = to_probability("0.5")                  # act
    assert result == expected                       # assert


def test_accepts_the_boundaries() -> None:
    assert (to_probability(0), to_probability(1)) == (0.0, 1.0)


def test_returns_a_float_not_a_string() -> None:
    assert isinstance(to_probability("0.25"), float)


# For now, run them by hand. Section 2 automates this.
for t in (test_accepts_a_valid_string, test_accepts_the_boundaries, test_returns_a_float_not_a_string):
    t()
print("3 tests passed (run by hand)")

3 tests passed (run by hand)


## 2. A minimal test runner

Running tests by hand has the same flaw as a bare `assert` script: the first failure hides the rest, and you have to remember to call each one. A runner fixes both. It needs perhaps fifteen lines:

- take a list of zero-argument functions;
- call each inside `try/except`, catching **any** exception (a failed `assert` *is* an `AssertionError`);
- record the traceback for failures;
- print one line per test and a summary, and return the counts so a cell can assert on them.

**Predict** what `run_tests` prints for a list containing two passing tests and one that does `assert 1 == 2`.

In [2]:
import traceback
from typing import Callable

def run_tests(tests: list[Callable[[], None]]) -> tuple[int, int]:
    """Run every test, report pass/fail, return (passed, failed). Never raises."""
    passed = failed = 0
    for test in tests:
        try:
            test()
        except Exception:
            failed += 1
            print(f"FAIL  {test.__name__}")
            print("      " + traceback.format_exc().strip().replace("\n", "\n      "))
        else:
            passed += 1
            print(f"ok    {test.__name__}")
    print(f"\n{passed} passed, {failed} failed")
    return passed, failed


def test_two_plus_two() -> None:
    assert 2 + 2 == 4


def test_deliberately_broken() -> None:
    # This one fails ON PURPOSE, to show the failure report. The notebook stays green.
    total = to_probability("0.4") + to_probability("0.4")
    assert total == 1.0, f"expected 1.0, got {total}"


passed, failed = run_tests([
    test_accepts_a_valid_string,
    test_two_plus_two,
    test_deliberately_broken,
])
assert (passed, failed) == (2, 1)     # the runner caught the failure instead of crashing

ok    test_accepts_a_valid_string
ok    test_two_plus_two
FAIL  test_deliberately_broken
      Traceback (most recent call last):
        File "<bonus_testing_for_ai.ipynb:2>", line 9, in run_tests
        File "<bonus_testing_for_ai.ipynb:2>", line 28, in test_deliberately_broken
      AssertionError: expected 1.0, got 0.8

2 passed, 1 failed


### Reading the failure report

`test_deliberately_broken` failed and the runner kept going. The report shows:

- the **test name**, so you can find it;
- the **assertion message** (`expected 1.0, got 0.8`) — always pass a message to `assert` in a test, because `0.4 + 0.4 == 0.8`, not `1.0`, and floating-point addition is exactly why (see section 5);
- the **line**, via the traceback.

From here on, every `test_*` in this notebook is expected to pass. Group related tests into one `run_tests([...])` call.

## 3. Table-driven tests: keep the case that failed

The earlier notebooks use this shape to check many inputs at once:

```python
for value in [0, 1, "0.25", "", "nan", -0.1]:
    ...  # assert something about each
```

It is compact, but when case 4 fails you get `AssertionError` with **no clue which value** caused it. A table-driven test fixes that: name each case, loop, and put the case identifier in the failure message (or collect them all).

**Predict** the output of the next cell: how many cases run, and does the reported failure (if any) name the offending input?

In [3]:
def check_cases(func, cases: list[tuple]) -> None:
    """cases: (case_id, args_tuple, expected). Reports every mismatch, then fails once."""
    failures = []
    for case_id, args, expected in cases:
        try:
            actual = func(*args)
        except Exception as exc:
            actual = f"<raised {type(exc).__name__}>"
        if actual != expected:
            failures.append(f"  {case_id}: to_probability{args!r} -> {actual!r}, expected {expected!r}")
    if failures:
        raise AssertionError("case failures:\n" + "\n".join(failures))


def test_to_probability_table() -> None:
    check_cases(to_probability, [
        ("zero",        (0,),      0.0),
        ("one",         (1,),      1.0),
        ("quarter_str", ("0.25",), 0.25),
        ("half_float",  (0.5,),    0.5),
    ])


def test_to_probability_rejections_table() -> None:
    # Every one of these SHOULD raise ValueError; the helper records the exact case if not.
    for case_id, bad in [("empty", ""), ("text", "nan"), ("negative", -0.1),
                         ("too_big", 1.1), ("none", None), ("nan_float", float("nan"))]:
        try:
            to_probability(bad)
        except ValueError:
            continue
        raise AssertionError(f"case {case_id}: to_probability({bad!r}) did not raise ValueError")


run_tests([test_to_probability_table, test_to_probability_rejections_table])

ok    test_to_probability_table
ok    test_to_probability_rejections_table

2 passed, 0 failed


(2, 0)

## 4. Testing that an error happens

"Rejects bad input with a clear error" is a specification, so it deserves a test. Asserting on an exception has a trap: a bare `try/except` that just `pass`es will also swallow the *wrong* exception, or a bug that raises nothing. A small context manager makes the intent exact.

**Predict** what `test_raises_helper_is_strict` proves about `raises(...)` when the wrong exception type is raised.

In [4]:
from contextlib import contextmanager

@contextmanager
def raises(expected: type[BaseException]):
    """Assert that the block raises `expected` (or a subclass). Anything else propagates."""
    try:
        yield
    except expected:
        return
    except BaseException as exc:               # wrong type -> make the test fail loudly
        raise AssertionError(
            f"expected {expected.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"expected {expected.__name__}, nothing was raised")


def test_rejects_non_numeric() -> None:
    with raises(ValueError):
        to_probability("banana")


def test_rejects_out_of_range() -> None:
    with raises(ValueError):
        to_probability(42)


def test_raises_helper_is_strict() -> None:
    # raises(KeyError) must NOT accept a ValueError -- it re-raises as AssertionError.
    with raises(AssertionError):
        with raises(KeyError):
            to_probability("banana")          # raises ValueError, not KeyError


run_tests([test_rejects_non_numeric, test_rejects_out_of_range, test_raises_helper_is_strict])

ok    test_rejects_non_numeric
ok    test_rejects_out_of_range
ok    test_raises_helper_is_strict

3 passed, 0 failed


(3, 0)

## 5. Testing floats and randomness

Two kinds of value break `==` in tests.

**Floating point.** `0.1 + 0.2 != 0.3`. Compare with a tolerance: `math.isclose` for scalars, `np.allclose` for arrays. This is exactly what notebook 06 does to check gradients against finite differences.

**Randomness.** A test must be deterministic. Two safe styles:

- **Seed and assert exact**: fix the seed, assert the sequence is reproducible.
- **Assert a property, not a value**: for a sampler you cannot pin, assert the *mean* is within tolerance of the target, or that all outputs are in range — never that draw 7 equals some literal.

**Predict** whether `0.1 + 0.2 == 0.3` and what `math.isclose(0.1 + 0.2, 0.3)` returns.

In [5]:
import random
import numpy as np

def test_float_sum_needs_a_tolerance() -> None:
    assert 0.1 + 0.2 != 0.3                        # the classic surprise
    assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)


def test_array_closeness() -> None:
    a = np.array([1.0, 2.0, 3.0])
    assert np.allclose(a / 3 * 3, a)               # not bitwise equal, but close


def test_seeded_sampler_is_reproducible() -> None:
    def draw(seed: int) -> list[float]:
        rng = random.Random(seed)
        return [rng.random() for _ in range(5)]
    assert draw(0) == draw(0)                      # same seed -> same sequence
    assert draw(0) != draw(1)


def test_sampler_property_not_value() -> None:
    rng = np.random.default_rng(12345)
    sample = rng.normal(loc=0.0, scale=1.0, size=10_000)
    assert abs(sample.mean()) < 0.05              # a property: mean is near 0
    assert 0.9 < sample.std() < 1.1               # never assert sample[7] == <literal>


run_tests([
    test_float_sum_needs_a_tolerance,
    test_array_closeness,
    test_seeded_sampler_is_reproducible,
    test_sampler_property_not_value,
])

ok    test_float_sum_needs_a_tolerance
ok    test_array_closeness
ok    test_seeded_sampler_is_reproducible
ok    test_sampler_property_not_value

4 passed, 0 failed


(4, 0)

## 6. Testing async code

An async test is a normal test that drives a coroutine with `asyncio.run(...)`. The interesting assertions for the notebook 05 client are behavioural:

- **retry count**: a transient failure is retried the right number of times, and no more;
- **concurrency ceiling**: never more than `limit` operations in flight at once.

Both are measured with a plain counter that the fake operation increments — no real network, no real clock.

**Predict** how many times `flaky` is called when it fails twice with `ConnectionError` and then succeeds, under `retry(..., attempts=3)`.

In [6]:
import asyncio
from course_utils import retry, bounded_map     # the implementations from notebook 05


def test_retry_stops_on_first_success() -> None:
    calls = 0
    async def flaky():
        nonlocal calls
        calls += 1
        if calls < 3:
            raise ConnectionError("temporary")
        return "ok"
    result = asyncio.run(retry(flaky, attempts=3, base_delay=0.0))
    assert result == "ok"
    assert calls == 3                              # 2 failures + 1 success, then it stops


def test_retry_gives_up_and_reraises() -> None:
    calls = 0
    async def always_fails():
        nonlocal calls
        calls += 1
        raise ConnectionError("down")
    with raises(ConnectionError):
        asyncio.run(retry(always_fails, attempts=2, base_delay=0.0))
    assert calls == 2                              # exactly `attempts`, not more


def test_retry_does_not_retry_permanent_errors() -> None:
    calls = 0
    async def bad_request():
        nonlocal calls
        calls += 1
        raise ValueError("invalid")
    with raises(ValueError):
        asyncio.run(retry(bad_request, attempts=5, base_delay=0.0))
    assert calls == 1                              # ValueError is not transient -> one attempt


def test_bounded_map_respects_the_ceiling() -> None:
    # Instrument concurrency by racing the real bounded_map against a counter.
    live = peak = 0
    original_sleep = asyncio.sleep
    async def tracking_sleep(delay):
        nonlocal live, peak
        live += 1; peak = max(peak, live)
        await original_sleep(delay)
        live -= 1
    asyncio.sleep = tracking_sleep
    try:
        out = asyncio.run(bounded_map(list(range(20)), limit=4))
    finally:
        asyncio.sleep = original_sleep
    assert out == [i * i for i in range(20)]
    assert peak <= 4                               # never more than `limit` in flight


run_tests([
    test_retry_stops_on_first_success,
    test_retry_gives_up_and_reraises,
    test_retry_does_not_retry_permanent_errors,
    test_bounded_map_respects_the_ceiling,
])

ok    test_retry_stops_on_first_success
ok    test_retry_gives_up_and_reraises
ok    test_retry_does_not_retry_permanent_errors
ok    test_bounded_map_respects_the_ceiling

4 passed, 0 failed


(4, 0)

## 7. Isolation: shared state makes order-dependent tests

Notebook 08's `append_note` tool appends to a module-level `recorded` list — a real side effect. If two tests share that list, the second test sees the first test's data, and the suite passes or fails depending on the **order** the tests run. That is a broken suite even when every assertion looks reasonable.

The fix is **setup/teardown**: give each test a fresh copy of the mutable state. In a runner you do this with a factory the test calls first; in `pytest` you do it with a *fixture*.

**Predict** whether `test_b_order_dependent` passes when run after `test_a_order_dependent`, and whether it would pass on its own.

In [7]:
# The trap: a shared mutable, mutated by tests.
shared_log: list[str] = []

def test_a_order_dependent() -> None:
    shared_log.append("from A")
    assert shared_log == ["from A"]               # true ONLY if nothing ran before

def test_b_order_dependent() -> None:
    shared_log.append("from B")
    assert shared_log == ["from B"]               # false: A already appended

print("--- shared mutable: order-dependent, one test corrupts the next ---")
shared_p, shared_f = run_tests([test_a_order_dependent, test_b_order_dependent])
assert (shared_p, shared_f) == (1, 1)          # the SECOND test fails purely because of the first


# The fix: a factory gives each test its own state. Nothing is shared.
def fresh_log() -> list[str]:
    return []

def test_a_isolated() -> None:
    log = fresh_log()
    log.append("from A")
    assert log == ["from A"]

def test_b_isolated() -> None:
    log = fresh_log()
    log.append("from B")
    assert log == ["from B"]

print("\n--- fresh state per test: order no longer matters ---")
p, f = run_tests([test_b_isolated, test_a_isolated])   # deliberately B before A
assert (p, f) == (2, 0)

--- shared mutable: order-dependent, one test corrupts the next ---
ok    test_a_order_dependent
FAIL  test_b_order_dependent
      Traceback (most recent call last):
        File "<bonus_testing_for_ai.ipynb:2>", line 9, in run_tests
        File "<bonus_testing_for_ai.ipynb:7>", line 10, in test_b_order_dependent
      AssertionError

1 passed, 1 failed

--- fresh state per test: order no longer matters ---
ok    test_b_isolated
ok    test_a_isolated

2 passed, 0 failed


## 8. What to test, and what to skip

You have limited time. Spend it on:

- **Boundaries** — `0` and `1` for a probability, empty and max-length for a string, `k = 0` and `k = len` for a top-k.
- **Invariants** — "the ring buffer's allocation never grows", "input order is preserved", "a frozen `Prediction` cannot be mutated".
- **Error paths** — every `raise` in the code should have a test that triggers it (section 4).
- **One wiring test** — the pieces integrate: a real config produces a working gateway.
- **Regressions** — when you fix a bug, add the test that would have caught it.

Skip:

- **Private helper functions** — test them through the public function that calls them, or every small code change breaks the tests for no reason.
- **Python itself** — you do not need a test that `+` adds.
- **The exact output of a random or model-driven step** — check a property instead (section 5).

"Code coverage" (the percentage of lines a test run touches) is a weak signal on its own: 100% coverage with no boundary or error-path checks is still a weak suite. A few sharp tests beat fifty that only try the easy path.

## 9. How this maps to `pytest`

The runner in this notebook is a teaching stand-in. Real projects use `pytest` (`pip install pytest`, then `pytest`). Every idea above has a direct equivalent:

| This notebook | `pytest` |
|---|---|
| `def test_x(): assert ...` | identical — `pytest` collects `test_*` functions in `test_*.py` files |
| `run_tests([...])` | the `pytest` command: discovers and runs everything, reports all failures |
| `check_cases(func, [(id, args, expected), ...])` | `@pytest.mark.parametrize("args,expected", [...], ids=[...])` |
| `with raises(ValueError):` | `with pytest.raises(ValueError):` |
| `math.isclose(a, b, rel_tol=...)` | `a == pytest.approx(b, rel=...)` |
| `fresh_log()` factory called in each test | `@pytest.fixture` that returns fresh state |
| `asyncio.run(coro())` in a test | `pytest-asyncio`'s `async def test_...` with `@pytest.mark.asyncio` |
| the "deliberately broken" demo | you never commit one; a red test blocks the merge |

The same tests from section 3, as a file `test_probability.py`:

```python
import math
import pytest
from mymodule import to_probability

@pytest.mark.parametrize("value,expected", [
    (0, 0.0), (1, 1.0), ("0.25", 0.25), (0.5, 0.5),
], ids=["zero", "one", "quarter_str", "half_float"])
def test_accepts(value, expected):
    assert to_probability(value) == pytest.approx(expected)

@pytest.mark.parametrize("bad", ["", "nan", -0.1, 1.1, None, float("nan")])
def test_rejects(bad):
    with pytest.raises(ValueError):
        to_probability(bad)
```

Run it with `pytest -q`. The output is the same information `run_tests` prints, with better diffs and no boilerplate.

## Project — A real test suite for an earlier project

Pick **one** project you have already implemented from an earlier notebook — the inference gateway (01), the streaming feature extractor (02), the constrained command decoder (03), or the resilient inference client (05) — and write a complete test suite for it.

### Requirements

1. **Structure.** Tests as `test_*` functions. Provide a `run_all()` that executes them through a runner like section 2's (or use `pytest` if you have it) and exits non-zero on any failure.
2. **Table-driven validation.** At least one parametrized test that drives a field through its full range of valid and invalid values, reporting the failing case by id.
3. **Error paths.** A test for every distinct exception the code can raise (unknown name, duplicate name, out-of-range confidence, wrong argument type, ...).
4. **Invariants.** At least two: e.g. "inputs are not mutated", "allocation does not grow", "input order preserved", "a duplicate `request_id` returns the cached result".
5. **Floats / randomness / async** — whichever apply to your chosen project, tested with tolerances / seeds / `asyncio.run` as in sections 5–6.
6. **Isolation.** No test depends on another test's side effects or on run order. Demonstrate this by running the suite in shuffled order and getting the same result.
7. **One integration test** that builds the component from a realistic config and exercises the happy path end to end.

**Acceptance criteria:** the suite runs headless and reports every failure (not just the first); each error path and at least two invariants are covered; no test is order-dependent; stochastic and async behaviour is tested by property/seed, not by brittle equality; and a one-paragraph note explains what the suite does **not** cover and why.

**Checks to run yourself**

- Run the suite, then run it again with the test list reversed (or `random.shuffle`d) — identical pass/fail counts.
- Break one line of the implementation under test; confirm exactly the tests that should fail do, and the report names them.
- Add a bug you have actually hit before, write the regression test, fix the bug.
- Count assertions per public method; any method with zero is a gap — justify it or fill it.
- If you have `pytest`: port three tests and confirm they pass unchanged.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse the runner ideas from this notebook (run_tests, check_cases, raises) and the
# implementation you are testing. If it lives in course_utils, import it, e.g.:
#     from course_utils import Registry, Prediction, AgentAction   # for the notebook 01 gateway
#
# 1. Write test_* functions: validation table, every error path, >=2 invariants,
#    floats/randomness/async as applicable, one integration test.
# 2. run_all(): execute them through a runner; return/exit non-zero on any failure.
# 3. Show order-independence: run the list reversed or shuffled, same result.
# 4. One paragraph: what this suite does NOT cover, and why.

def run_all() -> tuple[int, int]:
    raise NotImplementedError("Write and run the test suite for your chosen project")
